## 실험 04 — 과거 리드율과 시간순 예측

시리즈 시작 전에 알려진 기록만 사용한다. 여기서 time series는 **시간순 expanding-window 평가**이며 RNN/GRU로 모델을 바꾼다는 뜻은 아니다.

- 기존 7개 입력과, 최근 5세트의 `15분 양수 리드율` 및 `15분 1000골드 이상 리드율` 차이를 추가한 9개 입력을 비교한다.
- 과거 실험에서 사용한 첫 80% (316시리즈)를 개발 구간으로 사용하고 마지막 79개는 제외한다. 이미 살펴본 개발 자료의 비교이므로 독립 최종 성능으로 주장하지 않는다.
- 개발 구간에 3개의 확장 학습/미래 평가 fold를 구성한다. 같은 날짜는 같은 구간에 넣는다.
- 각 fold의 과거 부분 안에서 마지막 약 20%를 early stopping용으로 따로 나눈다. 바깥 미래 평가 label은 epoch 선택에 쓰지 않는다.
- 두 모델 모두 동일 구조(16→8), Adam 0.001, 최대 500 epochs, patience 30, seed 42/43/44, 양방향 확률 평균을 사용한다. 팀 뒤집기 학습 증강은 이번에는 추가하지 않는다.
- rolling 계산은 현재 세트를 shift로 제외하고 시리즈 첫 세트의 값을 선택한다. 앞선 평가 시리즈의 결과가 뒤 시리즈 feature에 반영되는 순차 예측 설정이며, 전체 기간을 시작 시점에 한꺼번에 예측하는 설정은 아니다.


In [1]:
from pathlib import Path
import sys
from copy import deepcopy
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score
from IPython.display import display

ts_root = Path('/Users/seungyunmok/Developer/LOL_ML')
if str(ts_root) not in sys.path:
    sys.path.insert(0, str(ts_root))
from backend.training.process_lck_data import load_many_lck_team_games
from backend.training.feature_pipeline import build_featured_games, build_match_dataset, DIFF_COLS

ts_raw = load_many_lck_team_games([
    ts_root / 'backend/data/raw' / f'{year}_LoL_esports_match_data_from_OraclesElixir.csv'
    for year in [2025, 2026]
])
ts_games = build_featured_games(ts_raw).sort_values(['teamname', 'date', 'gameid']).copy()
# 결측은 False로 바꾸지 않는다. 이 실험 데이터에는 결측이 없어야 한다.
assert ts_games['golddiffat15'].notna().all()
for name, threshold in [('lead15_rate', 0), ('strong_lead15_rate', 1000)]:
    indicator = (ts_games['golddiffat15'] > 0) if threshold == 0 else (ts_games['golddiffat15'] >= threshold)
    ts_games[name + '_indicator'] = indicator.astype(float)
    ts_games[name] = ts_games.groupby('teamname')[name + '_indicator'].transform(
        lambda s: s.shift(1).rolling(5, min_periods=1).mean()
    ).fillna(0.5)  # 과거 기록이 없는 경우의 공통 초기값. 실제 리그 빈도의 추정값은 아니다.

ts_matches = build_match_dataset(ts_games).sort_values(['date', 'series_key']).reset_index(drop=True)
ts_first = ts_games.sort_values(['date', 'gameid']).drop_duplicates(['series_key', 'teamname']).set_index(['series_key', 'teamname'])
ts_new_cols = []
for name in ['lead15_rate', 'strong_lead15_rate']:
    column = 'diff_' + name
    ts_new_cols.append(column)
    ts_matches[column] = [
        float(ts_first.loc[(row.series_key, row.team1), name] - ts_first.loc[(row.series_key, row.team2), name])
        for row in ts_matches.itertuples()
    ]
ts_matches['is_bo5'] = ts_matches['best_of'].eq(5).astype(float)
ts_dev = ts_matches.iloc[:int(len(ts_matches) * 0.8)].copy()
ts_dev['day'] = ts_dev['date'].dt.normalize()
ts_days = np.sort(ts_dev['day'].unique())
ts_groups = {'base_7': list(DIFF_COLS), 'lead_9': list(DIFF_COLS) + ts_new_cols}
print(f'개발 시리즈 {len(ts_dev)}개 / 제외한 마지막 구간 {len(ts_matches)-len(ts_dev)}개')
display(ts_dev[['date', *ts_new_cols]].head())

def ts_arrays(frame, columns, scale, reverse=False):
    values = frame[columns].to_numpy(dtype=float)
    if reverse:
        values = -values
    x = np.column_stack([scale.transform(values), frame['is_bo5'].to_numpy()])
    return torch.tensor(x, dtype=torch.float32)

def ts_probs(net, forward, reverse):
    return (torch.sigmoid(net(forward).squeeze(-1)) + 1 - torch.sigmoid(net(reverse).squeeze(-1))) / 2

ts_scores = []
ts_predictions = []
ts_fold_info = []
for fold, (past_idx, future_idx) in enumerate(TimeSeriesSplit(n_splits=3).split(ts_days), 1):
    past_days, future_days = ts_days[past_idx], ts_days[future_idx]
    inner_cut = int(len(past_days) * 0.8)
    fit = ts_dev[ts_dev['day'].isin(past_days[:inner_cut])]
    stop = ts_dev[ts_dev['day'].isin(past_days[inner_cut:])]
    future = ts_dev[ts_dev['day'].isin(future_days)]
    assert fit['date'].max() < stop['date'].min() < future['date'].min()
    ts_fold_info.append({'fold': fold, 'fit': len(fit), 'early_stop': len(stop), 'future': len(future),
                         'future_start': future['date'].min(), 'future_end': future['date'].max()})
    for variant, columns in ts_groups.items():
        scale = StandardScaler().fit(fit[columns].to_numpy(dtype=float))
        xf = ts_arrays(fit, columns, scale)
        xs, xsr = ts_arrays(stop, columns, scale), ts_arrays(stop, columns, scale, True)
        xt, xtr = ts_arrays(future, columns, scale), ts_arrays(future, columns, scale, True)
        yf = torch.tensor(fit['result'].to_numpy(), dtype=torch.float32)
        ys = torch.tensor(stop['result'].to_numpy(), dtype=torch.float32)
        for seed in [42, 43, 44]:
            torch.manual_seed(seed)
            net = nn.Sequential(nn.Linear(len(columns)+1,16), nn.ReLU(), nn.Linear(16,8), nn.ReLU(), nn.Linear(8,1))
            opt = torch.optim.Adam(net.parameters(), lr=0.001)
            loss_fn = nn.BCEWithLogitsLoss()
            best, saved, best_epoch, stale = float('inf'), None, 0, 0
            for epoch in range(1, 501):
                net.train(); opt.zero_grad()
                loss = loss_fn(net(xf).squeeze(-1), yf)
                loss.backward(); opt.step()
                net.eval()
                with torch.no_grad():
                    pstop = ts_probs(net, xs, xsr).clamp(1e-7, 1-1e-7)
                    stop_loss = nn.functional.binary_cross_entropy(pstop, ys).item()
                if stop_loss < best:
                    best, saved, best_epoch, stale = stop_loss, deepcopy(net.state_dict()), epoch, 0
                else:
                    stale += 1
                if stale >= 30:
                    break
            net.load_state_dict(saved); net.eval()
            with torch.no_grad():
                pred = ts_probs(net, xt, xtr).numpy()
            truth = future['result'].to_numpy()
            ts_scores.append({'fold':fold, 'variant':variant, 'seed':seed, 'n':len(future), 'epoch':best_epoch,
                              'log_loss':log_loss(truth,pred,labels=[0,1]), 'brier':brier_score_loss(truth,pred),
                              'accuracy':accuracy_score(truth,pred>=0.5)})
            for key, date, target, probability in zip(future['series_key'],future['date'],truth,pred):
                ts_predictions.append({'fold':fold,'variant':variant,'seed':seed,'series_key':key,'date':date,'y':int(target),'p':float(probability)})
    print(f'fold {fold} 완료')
ts_scores = pd.DataFrame(ts_scores)
ts_predictions = pd.DataFrame(ts_predictions)
print('기간 및 표본 수')
display(pd.DataFrame(ts_fold_info))
print('fold별 3개 seed 평균 (seed 반복은 독립 경기 추가가 아님)')
display(ts_scores.groupby(['fold','variant'])[['log_loss','brier','accuracy']].mean().round(4))
ts_overall = []
for (variant, seed), rows in ts_predictions.groupby(['variant','seed']):
    ts_overall.append({'variant':variant,'seed':seed,'n':len(rows),
                       'log_loss':log_loss(rows['y'],rows['p'],labels=[0,1]),
                       'brier':brier_score_loss(rows['y'],rows['p']),
                       'accuracy':accuracy_score(rows['y'],rows['p']>=0.5)})
ts_overall = pd.DataFrame(ts_overall)
print('미래 평가 행을 합친 성능: seed별 결과 및 평균')
display(ts_overall.round(4))
display(ts_overall.groupby('variant')[['log_loss','brier','accuracy']].agg(['mean','std']).round(4))
print('50% 확률 기준 log loss:', np.log(2), '/ Brier: 0.25')


개발 시리즈 316개 / 제외한 마지막 구간 79개


,date,diff_lead15_rate,diff_strong_lead15_rate
0,2025-01-15 08:09:30,0.0,0.0
1,2025-01-15 11:26:48,0.0,0.0
2,2025-01-16 08:09:06,0.0,0.0
3,2025-01-16 11:11:06,0.0,0.0
4,2025-01-17 08:08:25,0.0,0.0


fold 1 완료


fold 2 완료
fold 3 완료
기간 및 표본 수


,fold,fit,early_stop,future,future_start,future_end
0,1,64,20,84,2025-05-03 06:07:27,2025-08-14 10:09:25
1,2,132,36,70,2025-08-15 08:06:45,2026-02-07 10:09:22
2,3,198,40,78,2026-02-08 08:06:06,2026-05-16 08:05:55


...

log_loss   brier  accuracy
fold variant                            
1    base_7     0.6927  0.2499    0.4246
     lead_9     0.6925  0.2498    0.4563
2    base_7     0.7442  0.2711    0.4714
     lead_9     0.7071  0.2559    0.5095
3    base_7     0.6469  0.2273    0.6880
     lead_9     0.6713  0.2391    0.6923

미래 평가 행을 합친 성능: seed별 결과 및 평균


,variant,seed,n,log_loss,brier,accuracy
0,base_7,42,232,0.6763,0.2417,0.4957
1,base_7,43,232,0.7176,0.2591,0.5560
2,base_7,44,232,0.6846,0.2452,0.5302
3,lead_9,42,232,0.6892,0.2481,0.5690
4,lead_9,43,232,0.6849,0.2459,0.5086
5,lead_9,44,232,0.6951,0.2501,0.5776


log_loss           brier         accuracy        
            mean     std    mean     std     mean     std
variant                                                  
base_7    0.6928  0.0219  0.2487  0.0092   0.5273  0.0303
lead_9    0.6898  0.0051  0.2481  0.0021   0.5517  0.0376

50% 확률 기준 log loss: 0.6931471805599453 / Brier: 0.25
